## Training Scripts ipynb Version

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

from fly_traffic.config import ConnectomeConfig, PPOConfig, TrafficEnvConfig
from fly_traffic.connectome.artifact import ConnectomeArtifact
from fly_traffic.connectome.shuffled import (
    default_shuffled_connectome_path,
    load_or_create_shuffled_connectome,
)
from fly_traffic.training.ppo_trainer import PPOTrainer

from train_agents import copy_checkpoints, print_parameter_table, train_agent_worker

## Configurations

In [ ]:
dtime = datetime.now().strftime("%Y%m%d_%H%M%S")

AGENTS = ["fly", 'fly_shuffled', "mlp", "gru"] 
TOTAL_TIMESTEPS = PPOConfig.total_timesteps # default: 163840
NUM_ENVS = PPOConfig.num_envs # default: 8
SEED = PPOConfig.seed # default: 42
LR = PPOConfig.learning_rate # default: 0.0008
CONNECTOME_PATH = Path(ConnectomeConfig.output_path) 
CHECKPOINTS_DIR = Path(PPOConfig.checkpoints_dir)
RESUME = False
PARALLEL_MODELS = 1 # up to 4 models can be trained in parallel, depending on your CPU cores and RAM
SUFFIX = "" # suffix digunakan untuk membdakan nama model, misal "_ver_20230601_123456". Jika kosong, nama model akan default ke "{agents}_ppo.pt"

# --- Inisialisasi ---
conn_config = ConnectomeConfig(output_path=CONNECTOME_PATH)

artifact = None
if {"fly", "fly_shuffled"} & set(AGENTS):
    if not conn_config.output_path.exists():
        raise FileNotFoundError(
            f"Sub-connectome artifact not found at {conn_config.output_path}. "
            "Run scripts/extract_connectome.py first."
        )
    artifact = ConnectomeArtifact.load(conn_config.output_path)

print_parameter_table(AGENTS, artifact)

## Backup checkpoint

Salin `{agent}_ppo{SUFFIX}.pt` ke `backups/` dengan timestamp.

In [ ]:
backup_dir = CHECKPOINTS_DIR / "backups"
backup_dir.mkdir(parents=True, exist_ok=True)

copy_checkpoints(
    agents=AGENTS,
    checkpoints_dir=CHECKPOINTS_DIR,
    suffix=SUFFIX,
    backup_dir=backup_dir,
    timestamp=datetime.now().strftime("%Y%m%d_%H%M%S"),
)

## Training sequential 
1 loop for each agent, train one by one

**only executed if `PARALLEL_MODELS = 1`**

In [ ]:
if PARALLEL_MODELS < 2:
    trainer = PPOTrainer(
        ppo_config=PPOConfig(
            total_timesteps=TOTAL_TIMESTEPS,
            num_envs=NUM_ENVS,
            seed=SEED,
            checkpoints_dir=CHECKPOINTS_DIR,
        ),
        env_config=TrafficEnvConfig(),
        connectome_config=conn_config,
    )

    for model_type in AGENTS:
        model_artifact = artifact
        if model_type == "fly_shuffled":
            model_artifact = load_or_create_shuffled_connectome(
                connectome_path=conn_config.output_path,
                shuffled_path=default_shuffled_connectome_path(conn_config.output_path),
            )

        filename = f"{model_type}_ppo{SUFFIX}.pt"
        ckpt = CHECKPOINTS_DIR / filename
        resume_from = ckpt if (RESUME and ckpt.exists()) else None

        print(
            f"\n--- {model_type.upper()} PPO | resume={resume_from is not None} | "
            f"{TOTAL_TIMESTEPS} steps | lr={LR} ---"
        )
        trainer.train(
            model_type=model_type,
            artifact=model_artifact,
            verbose=True,
            checkpoint_filename=filename,
            resume_from=resume_from,
            learning_rate=LR,
        )

## Training paralel

Training multiple agents in parallel, each agent has its own trainer. Set `RESUME = True` to continue from checkpoint.

In [ ]:
if PARALLEL_MODELS >= 2:

    max_workers = min(PARALLEL_MODELS, len(AGENTS))

    with ProcessPoolExecutor(
        max_workers=max_workers, mp_context=mp.get_context("spawn")
    ) as executor:
        futures = {
            executor.submit(
                train_agent_worker,
                model_type,
                TOTAL_TIMESTEPS,
                NUM_ENVS,
                SEED,
                CONNECTOME_PATH,
                CHECKPOINTS_DIR,
                SUFFIX,
                RESUME,
                LR,
            ): model_type
            for model_type in AGENTS
        }
        for future in as_completed(futures):
            model_type = futures[future]
            future.result()  # raise jika worker gagal
            print(f"[MAIN] {model_type.upper()} selesai.")

## Benchmark & Evaluation

Evaluasi seluruh model yg udah dilatih + `FixedTimerAgent` dengan 4 skenario (`normal`, `asymmetric`, `rush_hour`, `stress_test`, 10 seed), uji farmakologi & lesi, ekspor JSON/Markdown, serta render grafik.

In [ ]:
from IPython.display import Image, display

from fly_traffic.agents.base import NeuralAgentAdapter
from fly_traffic.agents.baselines import load_policy_checkpoint
from fly_traffic.agents.fixed_timer import FixedTimerAgent
from fly_traffic.agents.fly_policy import FlyConnectomePolicy
from fly_traffic.config import BenchmarkConfig
from fly_traffic.evaluation.evaluator import AgentEvaluator
from fly_traffic.evaluation.visualizer import BenchmarkPlotter

BENCHMARK_OUTPUT_DIR = Path(BenchmarkConfig.output_dir)
BENCHMARK_LABELS = {
    "fly": "Fly Connectome PPO",
    "fly_shuffled": "Shuffled Connectome PPO",
    "gru": "PPO-GRU",
    "mlp": "PPO-MLP",
}

shuffled_artifact = (
    load_or_create_shuffled_connectome(
        connectome_path=conn_config.output_path,
        shuffled_path=default_shuffled_connectome_path(conn_config.output_path),
    )
    if "fly_shuffled" in AGENTS
    else None
)

In [ ]:
eval_agents = []
histories = {}
fly_policy = None

for model_type in AGENTS:
    ckpt_path = CHECKPOINTS_DIR / f"{model_type}_ppo{SUFFIX}.pt"
    if not ckpt_path.exists():
        print(f"Skipping missing checkpoint: {ckpt_path}")
        continue
    policy, payload = load_policy_checkpoint(
        path=ckpt_path,
        artifact=artifact if model_type == "fly" else None,
        shuffled_artifact=shuffled_artifact if model_type == "fly_shuffled" else None,
    )
    display_name = f"{BENCHMARK_LABELS[model_type]} ({ckpt_path.stem})"
    eval_agents.append(NeuralAgentAdapter(policy=policy, agent_name=display_name))
    if isinstance(payload.get("history"), dict):
        histories[display_name] = payload["history"]
    if model_type == "fly" and isinstance(policy, FlyConnectomePolicy):
        fly_policy = policy

eval_agents.append(FixedTimerAgent())

In [ ]:
evaluator = AgentEvaluator(
    benchmark_config=BenchmarkConfig(output_dir=BENCHMARK_OUTPUT_DIR),
    env_config=TrafficEnvConfig(),
)

scenario_results = evaluator.evaluate_all_agents(eval_agents)
pharmacology_results = (
    evaluator.evaluate_pharmacology_and_lesions(fly_policy)
    if fly_policy is not None
    else []
)

In [ ]:
BENCHMARK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_path, md_path = evaluator.save_reports(
    output_dir=BENCHMARK_OUTPUT_DIR,
    scenario_results=scenario_results,
    pharmacology_results=pharmacology_results,
)

plot_paths = []
if histories:
    p_curves = BENCHMARK_OUTPUT_DIR / "training_curves.png"
    BenchmarkPlotter.plot_training_curves(histories=histories, output_path=p_curves)
    plot_paths.append(p_curves)

p_scenarios = BENCHMARK_OUTPUT_DIR / "scenario_comparison.png"
BenchmarkPlotter.plot_scenario_comparison(results=scenario_results, output_path=p_scenarios)
plot_paths.append(p_scenarios)

if pharmacology_results:
    p_pharma = BENCHMARK_OUTPUT_DIR / "pharmacology_ablation.png"
    BenchmarkPlotter.plot_pharmacology_ablation(results=pharmacology_results, output_path=p_pharma)
    plot_paths.append(p_pharma)

print(f"Saved JSON report: {json_path}")
print(f"Saved Markdown table: {md_path}\n")
print(md_path.read_text(encoding="utf-8"))
for fig_path in plot_paths:
    display(Image(filename=str(fig_path)))